In [66]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

In [67]:
from openai import OpenAI

# Testing the OpenAI client
openai = OpenAI()

In [68]:
import os

from anthropic import Anthropic

# Testing the Anthropic client
anthropic = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [69]:
def llm(prompt: str) -> str:
    """
    Simple function to call the LLM and return the response with Anthropic API.
    """
    response = anthropic.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(
        block.text
        for block in response.content
        if getattr(block, "type", None) == "text"
    )

In [70]:
print(llm("Hello, how are you?"))

Hello! I'm doing well, thank you for asking. I'm here and ready to help with whatever you need. How can I assist you today?


In [71]:
question = "I just discovered the course. Can I still join?"
answer = llm(question)
print(answer)

I'd be happy to help, but I need a bit more information! I don't have context about which specific course you're referring to.

To give you the best answer, could you tell me:

- **Which course** are you asking about?
- **When does it start?**
- **Where did you find it?** (a website, school, platform, etc.)

Once you provide these details, I can give you better guidance on enrollment deadlines and


In [72]:
# Adding Context
context = """
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [73]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [74]:
answer = llm(prompt)
print(answer)

# Can I Still Join the Course?

Yes, you can still join the course! Here's what you need to know:

- **You can start learning immediately** - You don't need to wait for any confirmation. You can begin right away.
- **Registration is not mandatory** - You can start learning and submitting homework without formally registering. Registration is simply used to gauge interest before the course start date and is not checked against any registered list.
- **Certificate requirement


In [75]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [76]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1406

In [77]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [78]:
documents[1100]

{'id': '4b65d5542d',
 'course': 'llm-zoomcamp',
 'section': 'Module 5: Monitoring',
 'question': 'OperationalError when running python prep.py: psycopg2. OperationalError: could not translate host name "postgres" to address: No such host is known. How do I fix this issue?',
 'answer': 'To resolve this error, update the `.env` file:\n\n- Change the `POSTGRES_HOST` variable to `localhost`.\n\n```ini\nPOSTGRES_HOST=localhost\n```'}

## Search with Minsearch and a Simple RAG

In [79]:
from minsearch import Index

In [80]:
index = Index(text_fields=["question", "section", "answer"], keyword_fields=["course"])

In [81]:
index.fit(documents)

In [82]:
search_results = index.search(
    question,
    boost_dict={"question": 2},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=3,
)

In [83]:
def search(question: str, course="llm-zoomcamp"):
    return index.search(
        question,
        boost_dict={"question": 2},
        filter_dict={"course": course},
        num_results=5,
    )

In [84]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone proj

In [85]:
# Building a prompt with the search results
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know.
"""

In [86]:
USER_PROMPT = f"""
Question:
{question}

Context:
{context}
"""

In [87]:
# Function to build the context
def build_context(search_results):
    context = []
    for doc in search_results:
        context.append(doc["section"])
        context.append("Q: " + doc["question"])
        context.append("A: " + doc["answer"])
        context.append("")  # Add a blank line for better readability
    return "\n".join(context)

In [88]:
def build_prompt(question: str, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT.format(question=question, context=context)
    return prompt.strip()

In [89]:
prompt = build_prompt(question, search_results)

In [90]:
print(prompt)

Question:
I just discovered the course. Can I still join?

Context:

Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
